# Reproduce the fragmentation result

This notebook re-runs one experiment from the `pii-leak-benchmark` harness and checks what it
produces against the numbers published in the repository, field by field.

**Runtime > Run all**, then wait about two minutes. Nothing here needs an API key, an account,
or any paid service, and the experiment sends only loopback traffic. Installation downloads public dependencies.

What it measures: whether a PII detector still finds a value when that value is split across two
streaming events. One inspector looks at each chunk on its own; the other carries a small buffer
across the boundary. Everything else about them is identical.

Method, limits and what a green result does *not* prove:
<https://github.com/ninadphalak/LLM-Shield-Proxy/blob/main/website/docs/conformance/reproduce-fragmentation.md>


## 1. Get the code


In [ ]:
import os
import subprocess

subprocess.run(['git', 'clone', '--quiet', 'https://github.com/ninadphalak/LLM-Shield-Proxy.git'], check=True)
os.chdir('LLM-Shield-Proxy')
subprocess.run(['git', 'checkout', '--quiet', 'benchmark-v0.3.0'], check=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


## 2. Install the harness

Only the measuring instrument is installed. The proxy this project also ships is **not**
installed and is not used: the two inspectors being compared live inside the benchmark itself.


In [ ]:
import platform
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', './pii-leak-benchmark'], check=True)
print('python:', sys.version.split()[0])
print('platform:', platform.platform())


## 3. Run it

Two policies, about 35 seconds each. It will look idle while running.


In [ ]:
import tempfile

run_dir = tempfile.mkdtemp(prefix='benchmark-reproduction-')
subprocess.run([sys.executable, 'benchmarks/reproduce_fragmentation.py', '--out', run_dir], check=True)


## 4. Read the verdict

Expected: `chunk-local` leaks 0.125 when values arrive whole and 1.00 when they are split
(DeltaFrag 0.875). `bounded-retention` leaks 0.125 in both arms (DeltaFrag 0.00).

Every other field, including the corpus and inspector digests, must match the published reports.
Only timestamps, timings, the random port, and your OS and Python version are excluded.


In [ ]:
import json

s = json.load(open(os.path.join(run_dir, 'reproduction-summary.json')))
print('REPRODUCED:', s['reproduced'])
print('revision  :', s['environment']['source_revision'][:12])
for policy, c in s['comparisons'].items():
    print()
    print(policy, '->', 'matched' if c['reproduced'] else 'DID NOT MATCH')
    for label, v in c['headlines'].items():
        flag = '' if v['published'] == v['produced'] else '   <-- DIFFERS'
        print(f"   {label:22} {str(v['produced'])[:20]:>20}{flag}")
    for d in c['unexpected_differences']:
        print('   unexpected:', d['path'], d['published'], '->', d['produced'])


## 5. Send the result back

Run the cell below to download the three files, then send them back with the output above.

**Please report failures and disagreements.** A number that did not match, a step that broke, or
a reading of the result you think is wrong is worth more than a clean pass.

Report: <https://github.com/ninadphalak/LLM-Shield-Proxy/issues/new>


In [ ]:
from google.colab import files  # skip this cell if you are not on Colab

for name in ('chunk-local.json', 'bounded-retention.json', 'reproduction-summary.json'):
    files.download(os.path.join(run_dir, name))
